In [1]:
# Install required libraries
!pip install -q earthengine-api geemap

import ee
import geemap

# Authenticate and initialize Earth Engine
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

print("Google Earth Engine initialized successfully!")

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'F:\\Science Fair 2026 Project\\.venv\\Lib\\site-packages\\future\\backports\\email\\encoders.py'
Check the permissions.



ModuleNotFoundError: No module named 'geemap'

In [ ]:
# Define region over Khulna / Southwest Bangladesh flood zone
roi = ee.Geometry.BBox(89.48, 22.78, 89.62, 22.92)

# Filter Sentinel-1 SAR imagery (VV polarization)
collection = (
    ee.ImageCollection("COPERNICUS/S1_SAR")
    .filterBounds(roi)
    .filterDate('2024-06-01', '2024-09-30')  # Example monsoon timeframe
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarization', 'VV'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    .select('VV')
)

# Create a median composite image
sar_image = collection.median().clip(roi)
print("Sentinel-1 SAR image composite loaded.")

In [ ]:
# Define sample training points or use existing water layers (e.g., JRC Global Surface Water)
water_mask = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').gt(80).clip(roi)

# Combine SAR image with water ground-truth labels (1 = Water, 0 = Land)
training_image = sar_image.addBands(water_mask.rename('label'))

# Sample points from the image for training the classifier
training_samples = training_image.sample(
    region=roi,
    scale=30,
    numPixels=2000,
    seed=42,
    geometries=True
)

print("Training samples generated:", training_samples.size().getInfo())

In [ ]:
# Train a Random Forest classifier with 100 trees
bands = ['VV']
target = 'label'

classifier = ee.Classifier.smileRandomForest(numTrees=100).train(
    features=training_samples,
    classProperty=target,
    inputProperties=bands
)

# Classify the SAR image to extract water/flood extent
classified_water = sar_image.classify(classifier)

# Visualize the result using geemap
Map = geemap.Map(center=[22.8456, 89.5403], zoom=12)
Map.addLayer(sar_image, {'min': -25, 'max': -5}, 'Sentinel-1 SAR VV')
Map.addLayer(classified_water.updateMask(classified_water), {'palette': ['blue']}, 'Predicted Flood/Water Mask')
Map.addLayerControl()
Map

In [ ]:
# You can export your resulting flood classification map as an Earth Engine asset to load directly in your dashboard
export_task = ee.batch.export.image.toAsset(
    image=classified_water,
    description='Flood_Classification_Export',
    assetId='projects/your-project-id/assets/flood_classified_khulna',
    region=roi.coordinates().getInfo(),
    scale=30,
    maxPixels=1e9
)
export_task.start()
print("Export task started to Earth Engine asset storage.")